# Stakeholder Classification — Hypotheses 1–2

Rule-based classification of organizations into seven stakeholder groups, supplemented by manually curated exception lists.


In [ ]:
import pandas as pd

# 1. Load node data and manually curated exception lists
nodes_df = pd.read_csv("nodes.csv")

public_exceptions_df = pd.read_csv("public_organization_exceptions.csv")
private_exceptions_df = pd.read_csv("private_organization_exceptions.csv")
stakeholder_overrides_df = pd.read_csv("stakeholder_type_overrides.csv")

public_organization_exceptions = public_exceptions_df["Label"].dropna().tolist()
private_organization_exceptions = private_exceptions_df["Label"].dropna().tolist()
stakeholder_override_map = dict(zip(
    stakeholder_overrides_df["Label"],
    stakeholder_overrides_df["OverrideType"]
))

# 2. Define stopwords used only for keyword-based classification.
# Korean terms are retained because the organization labels in the source data are Korean.
stopwords = [
    "한국", "자원", "공사", "센터", "정보", "기관", "통합", "기술", "지원", "유일", "미래산업", "MOU",
    "연구", "개발", "조합", "운영", "혁신", "사회", "시스템", "플랫폼", "데이터", "서비스", "CES",
    "사업단", "본부", "소속", "네트워크", "스마트", "도시", "전략", "기반", "활용", "교육", "컨설팅"
]

# 3. Define keyword dictionaries for public- and private-sector organizations
public_keywords = [
    "부", "처", "청", "위원회", "감사원", "국회", "대통령실", "총리실", "국무조정실",
    "국정원", "청와대", "정부부처", "행정부", "공공기관운영위원회", "국가위원회", "중앙정부",
    "시청", "도청", "구청", "군청", "광역시청", "특별자치시청", "특별자치도청",
    "시의회", "도의회", "군의회", "자치구", "자치구청", "지방자치단체", "지방정부", "지자체", "자치단체",
    "행정복지센터", "공사", "공단", "재단", "센터", "진흥원", "지원단", "LH", "관리공단",
    "의료원", "국립", "정보원", "안전원", "공공기관", "공무원교육원", "공공재단", "지방공기업", "지방출자출연기관"
]

private_keywords = [
    "주식회사", "㈜", "기업", "은행", "통신", "상공회의소",
    "삼성", "현대", "현대차", "기아", "LG", "KT", "SK",
    "카카오", "네이버", "한화", "포스코", "롯데", "두산",
    "CJ", "GS", "LS", "효성", "대우", "엔씨소프트", "넷마블",
    "스타트업", "벤처", "프롭테크", "핀테크", "AI기업", "빅데이터기업",
    "컨설팅", "자문회사", "회계법인", "로펌", "건설사", "디벨로퍼"
]

# 4. Remove generic terms before keyword matching
def remove_stopwords(label):
    for stopword in stopwords:
        label = label.replace(stopword, "")
    return label.strip()

# 5. Classify each organization into one of seven stakeholder groups
def classify_stakeholder_type(label):
    # Priority 1: manually specified stakeholder-type overrides
    if label in stakeholder_override_map:
        return stakeholder_override_map[label]

    # Priority 2: public/private exception lists
    if label in public_organization_exceptions:
        return "Public Institution"
    if label in private_organization_exceptions:
        return "Private Sector"

    # Priority 3: keyword-based classification after label cleaning
    cleaned_label = remove_stopwords(label)

    if any(keyword in cleaned_label for keyword in public_keywords[:16]):
        return "Central Government"
    elif any(keyword in cleaned_label for keyword in public_keywords[16:32]):
        return "Local Government"
    elif any(keyword in cleaned_label for keyword in public_keywords[32:]):
        return "Public Institution"
    elif any(keyword in cleaned_label for keyword in [
        "대학교", "대학", "학교", "연구원", "연구소",
        "KAIST", "POSTECH", "GIST", "DGIST", "UNIST", "과학기술원"
    ]):
        return "Academia"
    elif any(keyword in cleaned_label for keyword in [
        "시민", "시민단체", "시민사회", "연대", "NGO", "비영리", "협동조합",
        "참여연대", "환경운동연합", "환경재단", "YWCA", "YMCA", "소비자연맹",
        "사회적협동조합", "사회적기업", "공익법인", "자원봉사센터", "공익활동지원센터"
    ]):
        return "Citizen Organization"
    elif any(keyword in cleaned_label for keyword in private_keywords):
        return "Private Sector"
    else:
        return "Other"

# 6. Apply the classification rule
nodes_df["stakeholder_type"] = nodes_df["Label"].apply(classify_stakeholder_type)

# 7. Save the classified node table
nodes_df.to_csv("nodes_classification.csv", index=False)


### Why were exception lists created?

Keyword-based rules alone were not sufficient to classify all smart-city organizations reliably. Some organization names contain ambiguous terms such as *foundation* or *center*, while others require contextual knowledge to distinguish public, private, academic, and local-government actors. Three manually curated exception files were therefore used alongside the rule-based classifier.

#### 1. `public_organization_exceptions.csv`

- Lists organizations that should be treated as **public institutions** even when their names contain ambiguous terms.
- Prevents predictable errors from the automatic keyword rules.
- Examples in the source data include the Korea Local Information Research & Development Institute and other public-sector bodies.

#### 2. `private_organization_exceptions.csv`

- Lists organizations whose names may resemble public institutions but that are **private companies or private foundations**.
- These cases are manually fixed as private-sector actors.

#### 3. `stakeholder_type_overrides.csv`

- Provides an explicit `Label` → `OverrideType` mapping for organizations whose stakeholder category is context-dependent or otherwise difficult to infer automatically.
- The override is evaluated before the keyword rules, giving manually reviewed cases the highest classification priority.

> **Note:** Korean organization names and Korean matching keywords are intentionally retained because they are part of the original source data. The analytical code, variable names, comments, file names, and output categories are presented in English for reproducibility and GitHub readability.
